# Credit Card Churn Modelling
## Logistic Regression and Random Forest

This notebook builds and compares two supervised classification models to predict customer attrition using the **cleaned, reduced output from the ETL notebook**.

### Workflow
1. Load and validate the cleaned dataset.
2. Define the target and exclude identifiers and leakage-prone columns.
3. Split the data into stratified training (80%) and test (20%) sets.
4. Build preprocessing/model pipelines for Logistic Regression and Random Forest.
5. Tune hyperparameters with cross-validation on the training set.
6. Select the model using **cross-validated F1-score**, then evaluate it once on the untouched test set.

### Modelling choices
- **Positive class:** `Attrited Customer` (the customer churned).
- **Primary selection metric:** F1-score, which balances precision and recall. This is useful because attrited customers are the minority class and both missed churners and false alarms matter. If the business priority is to catch as many potential churners as possible, recall could instead be selected as the primary metric.
- **Leakage prevention:** customer identifiers and columns whose names indicate that they encode a precomputed attrition prediction are excluded. The target is never included as a feature.
- **Preprocessing:** categorical features are one-hot encoded; numerical features are imputed. Logistic Regression also scales numerical features. All learned preprocessing is fitted inside the pipeline, preventing information from the test set leaking into training.
- **Test set discipline:** the test set is not used for hyperparameter tuning or model selection. It is used for final evaluation only.

## 1. Imports and configuration

Update `DATA_PATH` if your reduced CSV has a different name or is stored in another folder. The notebook first looks for `BankChurners_reduced.csv` in the current working directory.

In [6]:

import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
)

RANDOM_STATE = 42
DATA_PATH = Path("../Datasets/BankChurners_reduced.csv")

# The standard Kaggle BankChurners dataset uses these labels.
TARGET_COLUMN = "Attrition_Flag"
POSITIVE_LABEL = "Attrited Customer"
NEGATIVE_LABEL = "Existing Customer"

# Set to a list of known identifier columns if your ETL uses other names.
IDENTIFIER_COLUMNS = ["CLIENTNUM"]

# Columns derived from a precomputed attrition classifier are leakage-prone.
LEAKAGE_NAME_FRAGMENTS = ["naive_bayes_classifier"]

print("Working directory:", Path.cwd())
print("Expected data file:", DATA_PATH.resolve())

Working directory: /Users/mariaoanacucu/Documents/VS Code Projects/Credit_card_churn_analysis/jupyter_notebooks
Expected data file: /Users/mariaoanacucu/Documents/VS Code Projects/Credit_card_churn_analysis/Datasets/BankChurners_reduced.csv


## 2. Load and inspect the cleaned data

The modelling data should be the **reduced CSV produced by the ETL notebook**, not the raw Kaggle download. We inspect the shape, column names, missing values and target distribution before modelling.

In [7]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH.resolve()}. "
        "Update DATA_PATH in the configuration cell to point to your ETL output."
    )

df = pd.read_csv(DATA_PATH)

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
display(df.head())
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types and non-null counts:")
display(df.info())

print("\nMissing values by column:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))

if TARGET_COLUMN not in df.columns:
    raise KeyError(
        f"Target column '{TARGET_COLUMN}' was not found. "
        f"Available columns: {df.columns.tolist()}. "
        "Update TARGET_COLUMN if the ETL notebook renamed it."
    )

print("\nTarget distribution (counts):")
display(df[TARGET_COLUMN].value_counts(dropna=False).to_frame("count"))

print("\nTarget distribution (proportions):")
display(
    (df[TARGET_COLUMN].value_counts(normalize=True, dropna=False) * 100)
    .round(2).to_frame("percentage")
)

Rows: 10,127
Columns: 16


,Attrition_Flag,Customer_Age,Gender,Education_Level,Marital_Status,Income_Category,Card_Category,Months_on_book,Total_Relationship_Count,Months_Inactive_12_mon,Contacts_Count_12_mon,Total_Trans_Amt,Total_Trans_Ct,Credit_Limit,Total_Revolving_Bal,Avg_Utilization_Ratio
0,Existing Customer,45,M,High School,Married,$60K - $80K,Blue,39,5,1,3,1144,42,12691.0,777,0.061
1,Existing Customer,49,F,Graduate,Single,Less than $40K,Blue,44,6,1,2,1291,33,8256.0,864,0.105
2,Existing Customer,51,M,Graduate,Married,$80K - $120K,Blue,36,4,1,0,1887,20,3418.0,0,0.000
3,Existing Customer,40,F,High School,Unknown,Less than $40K,Blue,34,3,4,1,1171,20,3313.0,2517,0.760
4,Existing Customer,40,M,Uneducated,Married,$60K - $80K,Blue,21,5,1,0,816,28,4716.0,0,0.000



Column names:
['Attrition_Flag', 'Customer_Age', 'Gender', 'Education_Level', 'Marital_Status', 'Income_Category', 'Card_Category', 'Months_on_book', 'Total_Relationship_Count', 'Months_Inactive_12_mon', 'Contacts_Count_12_mon', 'Total_Trans_Amt', 'Total_Trans_Ct', 'Credit_Limit', 'Total_Revolving_Bal', 'Avg_Utilization_Ratio']

Data types and non-null counts:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10127 entries, 0 to 10126
Data columns (total 16 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Attrition_Flag            10127 non-null  object 
 1   Customer_Age              10127 non-null  int64  
 2   Gender                    10127 non-null  object 
 3   Education_Level           10127 non-null  object 
 4   Marital_Status            10127 non-null  object 
 5   Income_Category           10127 non-null  object 
 6   Card_Category             10127 non-null  object 
 7   Months_on_book            1

None


Missing values by column:


,missing_count
Attrition_Flag,0
Customer_Age,0
Gender,0
Education_Level,0
Marital_Status,0
Income_Category,0
Card_Category,0
Months_on_book,0
Total_Relationship_Count,0
Months_Inactive_12_mon,0



Target distribution (counts):


,count
Attrition_Flag,
Existing Customer,8500
Attrited Customer,1627



Target distribution (proportions):


,percentage
Attrition_Flag,
Existing Customer,83.93
Attrited Customer,16.07


## 3. Define the target and prepare features

The target is converted to a binary variable: attrited customers are `1`, and existing customers are `0`. The target column, identifier columns and any columns with names indicating a precomputed Naive Bayes attrition prediction are excluded from the predictors.

This cell also checks that both target classes are present. If your cleaned dataset uses different target labels, update `POSITIVE_LABEL` and `NEGATIVE_LABEL` above.

In [8]:
# Remove rows with a missing target; they cannot be used for supervised learning.
model_df = df.dropna(subset=[TARGET_COLUMN]).copy()

observed_labels = set(model_df[TARGET_COLUMN].astype(str).unique())
expected_labels = {POSITIVE_LABEL, NEGATIVE_LABEL}
if not expected_labels.issubset(observed_labels):
    raise ValueError(
        "The target labels do not match the configured labels. "
        f"Observed labels: {sorted(observed_labels)}. "
        f"Expected at least: {sorted(expected_labels)}. "
        "Update POSITIVE_LABEL / NEGATIVE_LABEL or adapt the target mapping."
    )

y = (model_df[TARGET_COLUMN].astype(str) == POSITIVE_LABEL).astype("int8")

# Detect leakage-prone columns by name, case-insensitively.
leakage_columns = [
    col for col in model_df.columns
    if any(fragment in col.lower() for fragment in LEAKAGE_NAME_FRAGMENTS)
]

identifier_columns_present = [
    col for col in IDENTIFIER_COLUMNS if col in model_df.columns
]

excluded_columns = sorted(set(
    [TARGET_COLUMN] + identifier_columns_present + leakage_columns
))

X = model_df.drop(columns=excluded_columns, errors="ignore").copy()

# Drop columns that contain no information.
constant_columns = [
    col for col in X.columns
    if X[col].nunique(dropna=False) <= 1
]
if constant_columns:
    X = X.drop(columns=constant_columns)

if X.shape[1] == 0:
    raise ValueError("No predictor columns remain after exclusions.")

print("Excluded target / identifier / leakage columns:", excluded_columns)
print("Dropped constant columns:", constant_columns)
print(f"Predictor count: {X.shape[1]}")
print(f"Target rows: {len(y):,}")
print(f"Attrited customers: {int(y.sum()):,} ({y.mean():.1%})")
print(f"Existing customers: {int((y == 0).sum()):,} ({(y == 0).mean():.1%})")

print("\nPredictor columns:")
print(X.columns.tolist())

print("\nPredictor data types:")
display(X.dtypes.astype(str).to_frame("dtype"))

Excluded target / identifier / leakage columns: ['Attrition_Flag']
Dropped constant columns: []
Predictor count: 15
Target rows: 10,127
Attrited customers: 1,627 (16.1%)
Existing customers: 8,500 (83.9%)

Predictor columns:
['Customer_Age', 'Gender', 'Education_Level', 'Marital_Status', 'Income_Category', 'Card_Category', 'Months_on_book', 'Total_Relationship_Count', 'Months_Inactive_12_mon', 'Contacts_Count_12_mon', 'Total_Trans_Amt', 'Total_Trans_Ct', 'Credit_Limit', 'Total_Revolving_Bal', 'Avg_Utilization_Ratio']

Predictor data types:


,dtype
Customer_Age,int64
Gender,object
Education_Level,object
Marital_Status,object
Income_Category,object
Card_Category,object
Months_on_book,int64
Total_Relationship_Count,int64
Months_Inactive_12_mon,int64
Contacts_Count_12_mon,int64


## 4. Stratified train/test split

We use an **80/20 split**, stratified by the binary target. Stratification helps preserve the overall proportion of attrited and existing customers in both sets.

All cross-validation and tuning happen on `X_train` and `y_train`. The test set remains untouched until final evaluation.

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

split_summary = pd.DataFrame({
    "rows": [len(y), len(y_train), len(y_test)],
    "attrited_count": [int(y.sum()), int(y_train.sum()), int(y_test.sum())],
    "attrited_percentage": [100 * y.mean(), 100 * y_train.mean(), 100 * y_test.mean()],
}, index=["Full data", "Training set", "Test set"])

display(split_summary.round(2))

,rows,attrited_count,attrited_percentage
Full data,10127,1627,16.07
Training set,8101,1302,16.07
Test set,2026,325,16.04


## 5. Build preprocessing and model pipelines

Column types are identified from the training data only:
- **Categorical columns:** missing values are filled with the most frequent category, then one-hot encoded.
- **Numerical columns:** missing values are filled with the median.
- **Logistic Regression:** numerical variables are additionally standardised.
- **Random Forest:** numerical variables are left on their original scale because tree-based models do not require scaling.

The preprocessing is embedded in each pipeline so that it is refitted independently within each cross-validation fold.

In [10]:
categorical_features = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
numerical_features = X_train.select_dtypes(include=["number"]).columns.tolist()

# Convert any remaining unusual data types to categorical strings rather than dropping them.
other_features = [
    col for col in X_train.columns
    if col not in categorical_features and col not in numerical_features
]
if other_features:
    X_train = X_train.copy()
    X_test = X_test.copy()
    for col in other_features:
        X_train[col] = X_train[col].astype("string")
        X_test[col] = X_test[col].astype("string")
    categorical_features.extend(other_features)

print("Categorical features:", categorical_features)
print("Numerical features:", numerical_features)

# Compatibility with newer and older scikit-learn versions.
try:
    one_hot_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
except TypeError:
    one_hot_encoder = OneHotEncoder(handle_unknown="ignore", sparse=True)

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", one_hot_encoder),
])

numeric_transformer_lr = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

numeric_transformer_rf = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

preprocessor_lr = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer_lr, numerical_features),
        ("categorical", categorical_transformer, categorical_features),
    ],
    remainder="drop",
)

preprocessor_rf = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer_rf, numerical_features),
        ("categorical", categorical_transformer, categorical_features),
    ],
    remainder="drop",
)

logistic_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_lr),
    ("classifier", LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )),
])

random_forest_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_rf),
    ("classifier", RandomForestClassifier(
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
])

print("Pipelines created.")

Categorical features: ['Gender', 'Education_Level', 'Marital_Status', 'Income_Category', 'Card_Category']
Numerical features: ['Customer_Age', 'Months_on_book', 'Total_Relationship_Count', 'Months_Inactive_12_mon', 'Contacts_Count_12_mon', 'Total_Trans_Amt', 'Total_Trans_Ct', 'Credit_Limit', 'Total_Revolving_Bal', 'Avg_Utilization_Ratio']
Pipelines created.


## 6. Hyperparameter optimisation with cross-validation

We use **5-fold stratified cross-validation** on the training set and optimise F1-score. F1-score balances precision (how many predicted churners really churned) and recall (how many actual churners were identified).

`class_weight="balanced"` gives the minority attrited class greater influence during fitting. This is a useful baseline for imbalanced classification, but it does not guarantee the best operational threshold; threshold selection can be explored separately after model comparison.

The search spaces below are intentionally moderate so they are practical to run in a typical notebook environment.

In [11]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

logistic_param_grid = {
    "classifier__C": [0.1, 1.0, 10.0],
    "classifier__solver": ["liblinear"],
}

random_forest_param_grid = {
    "classifier__n_estimators": [200, 400],
    "classifier__max_depth": [None, 10, 20],
    "classifier__min_samples_split": [2, 10],
    "classifier__min_samples_leaf": [1, 3],
    "classifier__max_features": ["sqrt"],
}

logistic_search = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid=logistic_param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    refit=True,
    return_train_score=False,
)

random_forest_search = GridSearchCV(
    estimator=random_forest_pipeline,
    param_grid=random_forest_param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    refit=True,
    return_train_score=False,
)

print("Tuning Logistic Regression...")
logistic_search.fit(X_train, y_train)

print("Tuning Random Forest...")
random_forest_search.fit(X_train, y_train)

print("\nBest Logistic Regression CV F1:", round(logistic_search.best_score_, 4))
print("Best Logistic Regression parameters:", logistic_search.best_params_)
print("\nBest Random Forest CV F1:", round(random_forest_search.best_score_, 4))
print("Best Random Forest parameters:", random_forest_search.best_params_)

Tuning Logistic Regression...
Tuning Random Forest...

Best Logistic Regression CV F1: 0.6149
Best Logistic Regression parameters: {'classifier__C': 1.0, 'classifier__solver': 'liblinear'}

Best Random Forest CV F1: 0.821
Best Random Forest parameters: {'classifier__max_depth': None, 'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 3, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 400}


## 7. Compare cross-validation results and select a model

Model selection is based on the **mean cross-validated F1-score** from the training set, not test-set performance. The selected estimator is already refitted on the complete training set by `GridSearchCV(refit=True)`.

In [12]:
cv_results = pd.DataFrame([
    {
        "model": "Logistic Regression",
        "best_cv_f1": logistic_search.best_score_,
        "best_parameters": logistic_search.best_params_,
    },
    {
        "model": "Random Forest",
        "best_cv_f1": random_forest_search.best_score_,
        "best_parameters": random_forest_search.best_params_,
    },
]).sort_values("best_cv_f1", ascending=False).reset_index(drop=True)

display(cv_results)

if logistic_search.best_score_ >= random_forest_search.best_score_:
    selected_name = "Logistic Regression"
    selected_search = logistic_search
else:
    selected_name = "Random Forest"
    selected_search = random_forest_search

best_model = selected_search.best_estimator_

print(f"Selected model based on training cross-validation F1: {selected_name}")
print(f"Best mean CV F1-score: {selected_search.best_score_:.4f}")

,model,best_cv_f1,best_parameters
0,Random Forest,0.820973,"{'classifier__max_depth': None, 'classifier__m..."
1,Logistic Regression,0.614933,"{'classifier__C': 1.0, 'classifier__solver': '..."


Selected model based on training cross-validation F1: Random Forest
Best mean CV F1-score: 0.8210


## 8. Final evaluation on the untouched test set

This is the first and only point at which we evaluate the selected model on the test set. We report:
- **Precision:** proportion of customers predicted to churn who actually churned.
- **Recall:** proportion of actual churners correctly identified.
- **F1-score:** harmonic mean of precision and recall.
- **ROC-AUC:** ability to rank attrited customers above existing customers across probability thresholds.
- **Accuracy:** proportion of all predictions that are correct.
- **Confusion matrix:** true negatives, false positives, false negatives and true positives.

The positive class is `Attrited Customer` (`1`). ROC-AUC uses predicted probabilities rather than the binary predictions.

In [13]:
# Predict once on the held-out test set.
y_pred = best_model.predict(X_test)

if hasattr(best_model, "predict_proba"):
    y_score = best_model.predict_proba(X_test)[:, 1]
else:
    y_score = best_model.decision_function(X_test)

test_metrics = {
    "Selected model": selected_name,
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, zero_division=0),
    "Recall": recall_score(y_test, y_pred, zero_division=0),
    "F1-score": f1_score(y_test, y_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, y_score),
}

metrics_table = pd.DataFrame(
    {"value": list(test_metrics.values())},
    index=list(test_metrics.keys()),
)
display(metrics_table.round(4))

print("\nClassification report (0 = Existing Customer, 1 = Attrited Customer):")
print(classification_report(
    y_test,
    y_pred,
    labels=[0, 1],
    target_names=[NEGATIVE_LABEL, POSITIVE_LABEL],
    zero_division=0,
))

cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[NEGATIVE_LABEL, POSITIVE_LABEL],
)
disp.plot(values_format="d")
plt.title(f"Confusion Matrix — {selected_name}")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

print("Confusion matrix layout: rows = actual class; columns = predicted class.")
print(f"TN = {cm[0, 0]}, FP = {cm[0, 1]}, FN = {cm[1, 0]}, TP = {cm[1, 1]}")

,value
Selected model,Random Forest
Accuracy,0.939289
Precision,0.821656
Recall,0.793846
F1-score,0.807512
ROC-AUC,0.973652



Classification report (0 = Existing Customer, 1 = Attrited Customer):
                   precision    recall  f1-score   support

Existing Customer       0.96      0.97      0.96      1701
Attrited Customer       0.82      0.79      0.81       325

         accuracy                           0.94      2026
        macro avg       0.89      0.88      0.89      2026
     weighted avg       0.94      0.94      0.94      2026



<Figure size 640x480 with 2 Axes>

Confusion matrix layout: rows = actual class; columns = predicted class.
TN = 1645, FP = 56, FN = 67, TP = 258


## 9. Interpretation and next steps

Use the held-out test metrics and confusion matrix to explain how the selected model performs for attrited customers. In a churn-retention setting:
- A **false negative** is an attrited customer the model failed to flag.
- A **false positive** is an existing customer incorrectly flagged as likely to churn.
- Higher recall may be valuable when missing a potential churner is costly, while precision matters when retention interventions are expensive or limited.

Do not choose a model based only on accuracy: with an imbalanced target, a model can achieve apparently good accuracy while missing many attrited customers.

### Suggested follow-up analysis
1. Compare the selected model with the other tuned model on the same held-out test set only if a pre-planned comparison is needed; avoid repeatedly changing the model based on test performance.
2. If the business priority is to identify as many churners as possible, consider selecting on recall instead of F1 in a future, clearly documented experiment.
3. Investigate the probability threshold using validation data or cross-validation predictions, not by repeatedly optimising against the test set.
4. Examine feature importance or model explanations, while noting that feature importance does not prove causality.
5. Document limitations, fairness considerations, data provenance, retention policy and GDPR-related safeguards in the wider project report.

### Reproducibility note
The random seed is fixed at `42`. Results may still vary slightly across software versions or computing environments. Record the dataset version, library versions and final parameter settings when reporting results.

## 10. Business interpretation

From a retention perspective, the model's value is not its overall accuracy but how well it helps the retention team spend a limited budget on the right customers. The selected model flags customers as likely to churn, and the confusion matrix translates directly into business outcomes: true positives are churners who can be targeted with a retention offer before they leave, false negatives are churners the model missed and who will be lost without intervention, and false positives are loyal customers who would have stayed anyway but may now receive an unnecessary (and potentially costly or irritating) incentive. Because churned customers are the minority, the default 0.5 threshold prioritises precision, which suits a business with a small retention budget that can only afford to contact high-confidence churners; if the business instead values catching as many at-risk customers as possible — for example because the cost of losing a customer far exceeds the cost of a retention offer — the threshold should be lowered (or recall selected as the primary metric) so that recall improves at the expense of precision. The ROC-AUC indicates how well the model can rank customers by churn risk, which is what matters operationally: even where the hard predictions at 0.5 are conservative, the predicted probabilities can be used to prioritise a ranked outreach list, targeting the highest-risk customers first until the retention budget is exhausted. Ultimately, the choice between the Logistic Regression and Random Forest models should be driven by which one better supports this ranked, budget-constrained targeting in production — with the simpler Logistic Regression offering easier explanation of churn drivers to business stakeholders — while the final decision threshold should be set by the retention team based on the relative cost of a missed churner versus a wasted incentive, not by the modelling default.